# TuneFlow — 01 Exploratory Data Analysis

This notebook explores the prepared dataset: users, songs, artists, interactions, sparsity, genre distribution and listening behavior.

**Prerequisites** (run from the project root):
```bash
python scripts/download_data.py --source sample   # or --source msd
python scripts/prepare_data.py --config sample.yaml
```
The same analysis is available as a script: `python scripts/eda.py` (figures land in `reports/figures`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from ml.preprocessing.pipeline import load_processed, load_processed_stats
from ml.preprocessing.filtering import compute_statistics

pd.options.display.float_format = '{:.4f}'.format
tables = load_processed()
songs, artists, interactions = tables['songs'], tables['artists'], tables['interactions']
print({k: len(v) for k, v in tables.items()})

## 1. Headline statistics
Users, songs, artists, interactions, sparsity and event-type distribution.

In [ ]:
stats = compute_statistics(songs, artists, interactions)
pd.DataFrame([stats]).T.rename(columns={0: 'value'})

## 2. User activity & song popularity distributions

In [ ]:
per_user = interactions.groupby('user_id').size().sort_values(ascending=False)
per_song = interactions.groupby('song_id').size().sort_values(ascending=False)

summary = pd.DataFrame({
    'users': per_user.describe(),
    'songs': per_song.describe(),
})
summary

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(per_user.clip(upper=per_user.quantile(0.99)), bins=40, color='#6366f1')
    axes[0].set_title('Interactions per user')
    axes[1].hist(np.log10(per_song.clip(lower=1)), bins=30, color='#22c55e')
    axes[1].set_title('Song popularity (log10)')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not installed')

## 3. Artist popularity & genre distribution

In [ ]:
song_artist = songs.set_index('song_id')['artist_id']
artist_names = songs.drop_duplicates('artist_id').set_index('artist_id')['artist_name']
artist_events = interactions.groupby('song_id').size().groupby(song_artist).sum().sort_values(ascending=False)
top_artists = artist_events.head(15).rename_axis('artist_id').reset_index(name='events')
top_artists['artist'] = top_artists['artist_id'].map(artist_names)
top_artists[['artist', 'events']]

In [ ]:
exploded = songs.explode('genres')
exploded = exploded[exploded['genres'].notna() & (exploded['genres'].astype(str) != '')]
genre_counts = exploded.groupby('genres').size().sort_values(ascending=False)
genre_counts.head(20)

## 4. Listening behavior over time & event mix

In [ ]:
if interactions['timestamp'].notna().any():
    ts = pd.to_datetime(interactions['timestamp'])
    daily = ts.dt.floor('D').value_counts().sort_index()
    hourly = ts.dt.hour.value_counts().sort_index()
    try:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
        axes[0].plot(daily.index, daily.values, color='#6366f1')
        axes[0].set_title('Daily activity')
        axes[1].bar(hourly.index, hourly.values, color='#f59e0b')
        axes[1].set_title('Hour of day')
        plt.tight_layout(); plt.show()
    except ImportError:
        pass
    display(daily.tail(7))
else:
    print('No timestamps in this dataset (e.g. MSD triplets).')

In [ ]:
interactions['event_type'].value_counts(normalize=True)

## 5. Audio feature landscape (when available)

In [ ]:
audio_cols = [c for c in ('energy','danceability','valence','acousticness','instrumentalness','speechiness','loudness','tempo') if c in songs.columns]
available = [c for c in audio_cols if pd.to_numeric(songs[c], errors='coerce').notna().any()]
if available:
    songs[available].describe().T
else:
    print('Dataset has no audio features - content model falls back to text/metadata features.')

## 6. Save all figures
`scripts/eda.py` produces the publication-ready versions of every plot above into `reports/figures/` plus `reports/eda_stats.json`.

```bash
python scripts/eda.py --config sample.yaml
```